In [2]:
import torch, json, os, random, numpy as np
from google.colab import drive
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/finsent"   # free Colab disconnects, so checkpoints go to Drive
os.makedirs(OUT, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
feat = ds["train"].features["label"]
id2label = {i: n for i, n in enumerate(feat.names)} if hasattr(feat, "names") else {0: "Bearish", 1: "Bullish", 2: "Neutral"}
label_words = [id2label[i].capitalize() for i in sorted(id2label)]

try:
    split = ds["train"].train_test_split(test_size=0.1, seed=SEED, stratify_by_column="label")
except ValueError:
    split = ds["train"].train_test_split(test_size=0.1, seed=SEED)
train_ds, dev_ds = split["train"], split["test"]   # dev is for training decisions
print(len(train_ds), len(dev_ds), label_words)    # the real validation split stays untouched

Mounted at /content/drive


README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

sent_train.csv:   0%|          | 0.00/859k [00:00<?, ?B/s]

sent_valid.csv:   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

8588 955 ['Bearish', 'Bullish', 'Neutral']


In [5]:
from huggingface_hub import login
from google.colab import userdata
from huggingface_hub import notebook_login

notebook_login()


In [1]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate scikit-learn

In [6]:
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"   # same model as the baseline
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token = tok.eos_token
tok.padding_side = "right"   # right for training; switch back to "left" for evaluation
PROMPT_DATE = "01 Jan 2026"   # fixed forever; the template otherwise inserts today's date

def build_prompt(text):
    msgs = [
        {"role": "system", "content": "You are a financial sentiment classifier."},
        {"role": "user", "content": f"Classify the sentiment of this finance tweet as exactly one of: {', '.join(label_words)}.\n\nTweet: {text}\n\nAnswer with one word."},
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   date_string=PROMPT_DATE)

print(build_prompt("test")[:200])   # must show "Today Date: 01 Jan 2026"
def encode(ex):
    p = tok(build_prompt(ex["text"]), add_special_tokens=False)["input_ids"]
    a = tok(label_words[ex["label"]] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return {"input_ids": p + a,
            "attention_mask": [1] * (len(p) + len(a)),
            "labels": [-100] * len(p) + a}   # loss only on the answer

train_tok = train_ds.map(encode, remove_columns=train_ds.column_names)
dev_tok = dev_ds.map(encode, remove_columns=dev_ds.column_names)
print("max length:", max(len(x) for x in train_tok["input_ids"]))
print(tok.decode(train_tok[0]["input_ids"]))

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 01 Jan 2026

You are a financial sentiment classifier.<|eot_id|><|start_header_id|>user<|


Map:   0%|          | 0/8588 [00:00<?, ? examples/s]

Map:   0%|          | 0/955 [00:00<?, ? examples/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


max length: 159
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 01 Jan 2026

You are a financial sentiment classifier.<|eot_id|><|start_header_id|>user<|end_header_id|>

Classify the sentiment of this finance tweet as exactly one of: Bearish, Bullish, Neutral.

Tweet: JinkoSolar Holding EPS beats by $0.056, beats on revenue

Answer with one word.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Bullish<|eot_id|>


In [7]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map="auto", dtype=torch.float16)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
for p in model.parameters():          # avoids the fp16 "unscale gradients" error
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [8]:
import shutil
shutil.rmtree(f"{OUT}/checkpoints", ignore_errors=True)

In [9]:
SMOKE = False
RUN = "r16_lr2e-4" + ("_smoke" if SMOKE else "")
CK = f"{OUT}/checkpoints_{RUN}"

args = TrainingArguments(
    output_dir=CK,
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    learning_rate=2e-4, num_train_epochs=2, max_steps=30 if SMOKE else -1,
    lr_scheduler_type="cosine", warmup_steps=30,
    fp16=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10, eval_strategy="steps", eval_steps=200,
    save_strategy="steps", save_steps=200, save_total_limit=2,
    report_to="none", seed=SEED,
)
collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100, return_tensors="pt")
trainer = Trainer(model=model, args=args, train_dataset=train_tok,
                  eval_dataset=dev_tok.select(range(300)), data_collator=collator)

resume = True if os.path.isdir(CK) and any(d.startswith("checkpoint-") for d in os.listdir(CK)) else None
trainer.train(resume_from_checkpoint=resume)

trainer.model.save_pretrained(f"{OUT}/adapter_{RUN}")
tok.save_pretrained(f"{OUT}/adapter_{RUN}")
json.dump(trainer.state.log_history, open(f"{OUT}/log_{RUN}.json", "w"))

Step,Training Loss,Validation Loss
200,0.118169,0.142986
400,0.137253,0.126825


Step,Training Loss,Validation Loss
200,0.118169,0.142986
400,0.137253,0.126825
600,0.051556,0.093916
800,0.061751,0.081891
1000,0.041418,0.081722
1074,0.039949,0.081641
